<a href="https://colab.research.google.com/github/l21141431-glitch/ISLP_labs/blob/main/WebScraping_Practica_darel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Celda 1: Instalacion de librerias
!pip install requests beautifulsoup4 lxml selenium pandas openpyxl
!apt-get update
!apt-get install -y chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 510.3/510.3 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.6/131.6 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,941 kB]
Hit:9 https://ppa.launchpadcontent.net/dea

In [3]:
# Verificar instalacion
import requests
from bs4 import BeautifulSoup
from selenium import webdriver
import pandas as pd
print('Todas las librerias instaladas correctamente')

Todas las librerias instaladas correctamente


## Paso 2


In [4]:
# Celda 2: Primera solicitud HTTP
import requests
from bs4 import BeautifulSoup

In [6]:
# Realizar solicitud GET al sitio
url = 'http://books.toscrape.com'
response = requests.get(url)

In [8]:
# Verificar que la solicitud fue exitosa
print(f'Status Code: {response.status_code}') # 200 = exito
print(f'Encoding: {response.encoding}')
print(f'Tamano del HTML: {len(response.text)} caracteres')

Status Code: 200
Encoding: ISO-8859-1
Tamano del HTML: 51294 caracteres


In [10]:
# Parsear el HTML con BeautifulSoup
soup = BeautifulSoup(response.text, 'lxml')

In [12]:
# Ver el titulo de la pagina
print(f'Titulo: {soup.title.string}')

Titulo: 
    All products | Books to Scrape - Sandbox



## Paso 3: extraer Datos de un solo Libro

In [14]:
# Celda 3: Extraer datos del primer libro
# Encontrar el primer articulo de producto
primer_libro = soup.find('article', class_='product_pod')

In [17]:
# Extraer titulo (esta en el atributo 'title' del enlace dentro de <h3>)
titulo = primer_libro.h3.a['title']
print(f'Titulo: {titulo}')

Titulo: A Light in the Attic


In [19]:
# Extraer precio
precio = primer_libro.find('p', class_='price_color').text
print(f'Precio: {precio}')

Precio: Â£51.77


In [21]:
calificacion = primer_libro.find('p', class_='star-rating')['class'][1]
print(f'Calificacion: {calificacion}')

Calificacion: Three


In [22]:
# Extraer disponibilidad
disponible = primer_libro.find(&#39;p&#39;, class_=&#39;instock availability&#39;)
stock = disponible.text.strip() if disponible else &#39;Sin info&#39;
print(f&#39;Disponibilidad: {stock}&#39;)

SyntaxError: invalid syntax (1680407430.py, line 2)

In [23]:
# Procesar el precio a formato numerico
precio_numerico = float(precio.replace('Â£', ''))
print(f'Precio numerico: {precio_numerico}')

Precio numerico: 51.77


## Paso 4 extraer todos los libros

In [25]:
# Celda 4: Extraer todos los libros de la pagina 1
libros = soup.find_all('article', class_='product_pod')
print(f'Libros encontrados en pagina 1: {len(libros)}')


Libros encontrados en pagina 1: 20


In [30]:
datos = []
for libro in libros:
    titulo = libro.h3.a['title']
    precio_texto = libro.find('p', class_='price_color').text
    # Limpiar precio: quitar simbolo y convertir a float
    precio = float(precio_texto.replace('Â£', '').strip())
    calificacion = libro.find('p', class_='star-rating')['class'][1]
    stock = libro.find('p', class_='instock availability')
    disponible = 'Si' if stock and 'In stock' in stock.text else 'No'
    datos.append({'Titulo': titulo, 'Precio': precio, 'Calificacion': calificacion, 'Disponible': disponible})

In [46]:
# This cell is redundant and was causing inconsistent keys in the 'datos' list.
# It has been removed as 'todos_los_libros' is now the primary data source.

In [45]:
# Mostrar primeros 5 resultados de todos los libros
import pandas as pd
df_todos_libros = pd.DataFrame(todos_los_libros)
print(df_todos_libros.head())
print(f'\nTotal registros: {len(df_todos_libros)}')


                                  Titulo  Precio Calificacion Disponible
0                   A Light in the Attic   51.77        Three         Si
1                     Tipping the Velvet   53.74          One         Si
2                             Soumission   50.10          One         Si
3                          Sharp Objects   47.82         Four         Si
4  Sapiens: A Brief History of Humankind   54.23         Five         Si

Total registros: 1000


## Paso 5 scraping con paginacion

In [35]:
# Celda 5: Scraping de TODAS las paginas (50 paginas)
import time

In [37]:
todos_los_libros = []
base_url = 'http://books.toscrape.com/catalogue/page-{}.html'


In [42]:
for pagina in range(1, 51): # 50 paginas
    url = base_url.format(pagina)
    response = requests.get(url)

    if response.status_code != 200:
        print(f'Error al acceder a la página {pagina}: {response.status_code}')
        continue

    soup = BeautifulSoup(response.text, 'lxml')
    libros_pagina = soup.find_all('article', class_='product_pod')

    for libro in libros_pagina:
        titulo = libro.h3.a['title']
        precio_texto = libro.find('p', class_='price_color').text
        # Limpiar precio: quitar simbolo y convertir a float
        precio = float(precio_texto.replace('Â£', '').strip())
        calificacion = libro.find('p', class_='star-rating')['class'][1]
        stock_element = libro.find('p', class_='instock availability')
        disponible = 'Si' if stock_element and 'In stock' in stock_element.text else 'No'
        todos_los_libros.append({'Titulo': titulo, 'Precio': precio, 'Calificacion': calificacion, 'Disponible': disponible})

    print(f'Página {pagina} procesada. Total de libros recolectados: {len(todos_los_libros)}')
    time.sleep(1) # Pequeña pausa para evitar sobrecargar el servidor

Página 1 procesada. Total de libros recolectados: 20
Página 2 procesada. Total de libros recolectados: 40
Página 3 procesada. Total de libros recolectados: 60
Página 4 procesada. Total de libros recolectados: 80
Página 5 procesada. Total de libros recolectados: 100
Página 6 procesada. Total de libros recolectados: 120
Página 7 procesada. Total de libros recolectados: 140
Página 8 procesada. Total de libros recolectados: 160
Página 9 procesada. Total de libros recolectados: 180
Página 10 procesada. Total de libros recolectados: 200
Página 11 procesada. Total de libros recolectados: 220
Página 12 procesada. Total de libros recolectados: 240
Página 13 procesada. Total de libros recolectados: 260
Página 14 procesada. Total de libros recolectados: 280
Página 15 procesada. Total de libros recolectados: 300
Página 16 procesada. Total de libros recolectados: 320
Página 17 procesada. Total de libros recolectados: 340
Página 18 procesada. Total de libros recolectados: 360
Página 19 procesada. To

In [43]:
# Este bloque fue movido e integrado en la celda del bucle principal (7X30UxVop518)

In [47]:
soup = BeautifulSoup(response.text, 'lxml')
libros = soup.find_all('article', class_='product_pod')

In [48]:
for libro in libros:
titulo = libro.h3.a[&#39;title&#39;]
precio_texto = libro.find(&#39;p&#39;, class_=&#39;price_color&#39;).text
precio = float(precio_texto.replace(&#39;\u00a3&#39;, &#39;&#39;).strip())
calificacion = libro.find(&#39;p&#39;, class_=&#39;star-rating&#39;)[&#39;class&#39;][1]
stock = libro.find(&#39;p&#39;, class_=&#39;instock availability&#39;)
disponible = &#39;Si&#39; if stock and &#39;In stock&#39; in stock.text else &#39;No&#39;

IndentationError: expected an indented block after 'for' statement on line 1 (3035408221.py, line 2)

In [49]:
display(df_todos_libros.describe(include='all'))

,Titulo,Precio,Calificacion,Disponible
count,1000,1000.00000,1000,1000
unique,999,NaN,5,1
top,The Star-Touched Queen,NaN,One,Si
freq,2,NaN,226,1000
mean,NaN,35.07035,NaN,NaN
std,NaN,14.44669,NaN,NaN
min,NaN,10.00000,NaN,NaN
25%,NaN,22.10750,NaN,NaN
50%,NaN,35.98000,NaN,NaN
75%,NaN,47.45750,NaN,NaN


In [52]:
# This cell is redundant as the `todos_los_libros` list was already fully populated by the multi-page scraping loop.

In [54]:
import pandas as pd # Ensure pandas is imported
# Pausa de cortesia entre solicitudes (buena practica)
# The 'time.sleep(0.5)' and 'if' block below are likely misplaced here,
# as this cell is not within the main scraping loop (cell 7X30UxVop518)
# where 'pagina' is iterated.
# However, to fix the syntax error as requested:
# time.sleep(0.5) # This sleep is out of context here.
# if pagina % 10 == 0:
#     print(f'Paginas procesadas: {pagina}/50') # Corrected indentation

# Assuming the intent was to display the final DataFrame after all scraping is done:
df_completo = pd.DataFrame(todos_los_libros)
print(f'\nTotal de libros extraidos: {len(df_completo)}')
print(df_completo.head(10))


Total de libros extraidos: 1001
                                              Titulo  Precio Calificacion  \
0                               A Light in the Attic   51.77        Three   
1                                 Tipping the Velvet   53.74          One   
2                                         Soumission   50.10          One   
3                                      Sharp Objects   47.82         Four   
4              Sapiens: A Brief History of Humankind   54.23         Five   
5                                    The Requiem Red   22.65          One   
6  The Dirty Little Secrets of Getting Your Dream...   33.34         Four   
7  The Coming Woman: A Novel Based on the Life of...   17.93        Three   
8  The Boys in the Boat: Nine Americans and Their...   22.60         Four   
9                                    The Black Maria   52.15          One   

  Disponible titulo  precio_gbp calificacion disponible  pagina_origen  
0         Si    NaN         NaN          NaN  

## Paso 6 Guardar datos

In [55]:
# Celda 6: Guardar en CSV y Excel

In [57]:
# Guardar en CSV
df_completo.to_csv('libros_scrapeados.csv', index=False, encoding='utf-8-sig')
print('Archivo CSV guardado: libros_scrapeados.csv')

Archivo CSV guardado: libros_scrapeados.csv


In [59]:
# Guardar en Excel
df_completo.to_excel('libros_scrapeados.xlsx', index=False)
print('Archivo Excel guardado: libros_scrapeados.xlsx')

Archivo Excel guardado: libros_scrapeados.xlsx


In [61]:
# Estadisticas basicas del dataset
print(f'\n--- Resumen del Dataset ---')
print(f'Registros totales: {len(df_completo)}')
print(f'Columnas: {list(df_completo.columns)}')
print(f'Precio promedio: £{df_completo["precio_gbp"].mean():.2f}')
print(f'Precio maximo: £{df_completo["precio_gbp"].max():.2f}')
print(f'Distribucion de calificaciones:')
print(df_completo['calificacion'].value_counts())


--- Resumen del Dataset ---
Registros totales: 1001
Columnas: ['Titulo', 'Precio', 'Calificacion', 'Disponible', 'titulo', 'precio_gbp', 'calificacion', 'disponible', 'pagina_origen']
Precio promedio: £26.08
Precio maximo: £26.08
Distribucion de calificaciones:
calificacion
Five    1
Name: count, dtype: int64


In [63]:
# Descargar archivo (en Colab)
from google.colab import files
files.download('libros_scrapeados.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>